In [0]:
df_hosp_a = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/hospedagens/vianorte/*.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

df_hosp_b = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/hospedagens/atlas/*.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

In [0]:
display(df_hosp_a)
display(df_hosp_b)

In [0]:
df_hosp_a.printSchema()
df_hosp_b.printSchema()

df_hosp_a.columns
df_hosp_b.columns

In [0]:
from pyspark.sql.functions import col, lit

In [0]:
df_hosp_a_padrao = df_hosp_a.select(
    col("Voucher").alias("id_hospedagem"),
    col("Hospede").alias("viajante"),
    col("Check-in").alias("data_checkin"),
    col("Check-out").alias("data_checkout"),
    col("Centro Custo").alias("centro_custo_origem"),
    col("Cidade").alias("cidade"),
    col("Total Hospedagem").alias("valor"),
    col("Status").alias("situacao")
).withColumn(
    "agencia",
    lit("ViaNorte")
)

In [0]:
df_hosp_b_padrao = df_hosp_b.select(
    col("Booking").alias("id_hospedagem"),
    col("Nome Passageiro").alias("viajante"),
    col("Entrada").alias("data_checkin"),
    col("Saida").alias("data_checkout"),
    col("Unidade").alias("centro_custo_origem"),
    col("Municipio").alias("cidade"),
    col("Valor NF").alias("valor"),
    col("Situacao").alias("situacao")
).withColumn(
    "agencia",
    lit("Atlas Corporate")
)

In [0]:
df_hosp_a_padrao.columns == df_hosp_b_padrao.columns

In [0]:
df_hosp_a_padrao.count()
df_hosp_b_padrao.count()

In [0]:
df_hosp_a_padrao = df_hosp_a_padrao.filter(
    col("id_hospedagem").isNotNull()
)

df_hosp_b_padrao = df_hosp_b_padrao.filter(
    col("id_hospedagem").isNotNull()
)

In [0]:
print(df_hosp_a_padrao.count())
print(df_hosp_b_padrao.count())

In [0]:
df_hospedagens = df_hosp_a_padrao.unionByName(
    df_hosp_b_padrao
)

In [0]:
df_hospedagens.count()

In [0]:
display(df_hospedagens)

In [0]:
from pyspark.sql.functions import regexp_replace

In [0]:
df_hospedagens = df_hospedagens.withColumn(
    "valor_numerico",
    regexp_replace(
        regexp_replace(
            regexp_replace(
                col("valor"),
                "R\\$ ",
                ""
            ),
            "\\.",
            ""
        ),
        ",",
        "."
    ).cast("decimal(18,2)")
)

In [0]:
df_hospedagens.select(
    "valor",
    "valor_numerico"
).show(20, truncate=False)

In [0]:
df_hospedagens.printSchema()

In [0]:
from pyspark.sql.functions import datediff

In [0]:
df_hospedagens = df_hospedagens.withColumn(
    "qtd_noites",
    datediff(
        col("data_checkout"),
        col("data_checkin")
    )
)

In [0]:
df_hospedagens.select(
    "viajante",
    "data_checkin",
    "data_checkout",
    "qtd_noites"
).show(20, truncate=False)

In [0]:
df_hospedagens = df_hospedagens.withColumn(
    "tipo_despesa",
    lit("Hospedagem")
)

In [0]:
df_hospedagens.select(
    "id_hospedagem",
    "tipo_despesa"
).show(10)

In [0]:
df_map = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/referencias/centro_custo/Centro_Custo_Map.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

In [0]:
df_hospedagens = (
    df_hospedagens.alias("h")
    .join(
        df_map.alias("m"),
        (
            (col("h.agencia") == col("m.Agencia")) &
            (col("h.centro_custo_origem") == col("m.Valor_Origem"))
        ),
        "left"
    )
    .select(
        "h.*",
        col("m.Centro_Custo_Padrao").alias("centro_custo"),
        col("m.Diretoria").alias("diretoria")
    )
)

In [0]:
df_hospedagens.select(
    "agencia",
    "centro_custo_origem",
    "centro_custo",
    "diretoria"
).show(30, truncate=False)

In [0]:
df_hospedagens.filter(
    col("centro_custo").isNull()
).count()

In [0]:
df_hospedagens.count()

In [0]:
from pyspark.sql.functions import month, when

In [0]:
df_hospedagens = df_hospedagens.withColumn(
    "trimestre",
    when(
        month(col("data_checkin")).between(1, 3),
        "Primeiro Trimestre"
    )
    .when(
        month(col("data_checkin")).between(4, 6),
        "Segundo Trimestre"
    )
    .when(
        month(col("data_checkin")).between(7, 9),
        "Terceiro Trimestre"
    )
    .otherwise(
        "Quarto Trimestre"
    )
)

In [0]:
df_hospedagens.groupBy("trimestre").count().show()

In [0]:
df_hospedagens.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("viagens_teste.silver.hospedagens")

In [0]:
df_hosp_teste = spark.table(
    "viagens_teste.silver.hospedagens"
)

display(df_hosp_teste)

In [0]:
print("Total de hospedagens:", df_hospedagens.count())

print(
    "Centros de custo não mapeados:",
    df_hospedagens.filter(
        col("centro_custo").isNull()
    ).count()
)